In [69]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

USE_MANUAL_5050_TEST = False   # True = test 50/50, False = stratified kfold

# Caricamento dati

In [70]:
FILE_PATH = Path("/Users/francesco/Tesi/BC-ML4/dataset/cleaned")
df = pd.read_csv(FILE_PATH / "ambl_lesions.csv")

df["PR_class"] = (pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1).astype(int)
df = df.dropna(subset=["PR_class"])

print("Totale campioni:", len(df))
print("Distribuzione PR_class:")
print(df["PR_class"].value_counts())


Totale campioni: 82
Distribuzione PR_class:
PR_class
0    46
1    36
Name: count, dtype: int64


# Target

In [71]:
drop_cols = [
    "Patient ID","lesion idx","tumor/benign",
    "GRADE","isTN","Breast",
    "ER [SII]","PR [SII]","HER2 [SII]",
    "PR_class"
]

groups = df["Patient ID"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["PR_class"]

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean())


# Split 50/50

In [72]:
def create_manual_5050_splits(X, y, n_splits=5, random_state=42):
    np.random.seed(random_state)

    idx0 = y[y == 0].index.to_numpy()
    idx1 = y[y == 1].index.to_numpy()

    np.random.shuffle(idx0)
    np.random.shuffle(idx1)

    n_min = min(len(idx0), len(idx1))
    n_test = n_min // n_splits

    splits = []
    for k in range(n_splits):
        test0 = idx0[k*n_test:(k+1)*n_test]
        test1 = idx1[k*n_test:(k+1)*n_test]

        test_idx = np.concatenate([test0, test1])
        train_idx = np.setdiff1d(np.arange(len(y)), test_idx)

        splits.append((train_idx, test_idx))

    return splits


# Bilanciamento manuale

In [73]:
def balance_test_sets_post_hoc(cv_splits, y):
    """
    Bilancia i test set a 50/50 post-hoc.
    I campioni in eccesso vengono spostati al training set.
    """
    balanced_splits = []
    
    for fold_num, (train_idx, test_idx) in enumerate(cv_splits, 1):
        y_test = y.iloc[test_idx]
        
        # Indici per classe
        mask_0 = (y_test == 0).values
        mask_1 = (y_test == 1).values
        
        idx_0 = test_idx[mask_0]
        idx_1 = test_idx[mask_1]
        
        n_0 = len(idx_0)
        n_1 = len(idx_1)
        n_min = min(n_0, n_1)
        
        print(f"Fold {fold_num}:")
        print(f"  PRIMA  - Test: Classe 0={n_0}, Classe 1={n_1}")
        
        # Campiona n_min per classe
        np.random.seed(42 + fold_num)
        idx_0_sampled = np.random.choice(idx_0, size=n_min, replace=False)
        idx_1_sampled = np.random.choice(idx_1, size=n_min, replace=False)
        
        # Test bilanciato
        test_idx_balanced = np.concatenate([idx_0_sampled, idx_1_sampled])
        
        # Campioni non usati → training
        unused = np.setdiff1d(test_idx, test_idx_balanced)
        train_idx_new = np.concatenate([train_idx, unused])
        
        print(f"  DOPO   - Test: Classe 0={n_min}, Classe 1={n_min} (totale={n_min*2})")
        print(f"         - Train: {len(train_idx_new)} campioni\n")
        
        balanced_splits.append((train_idx_new, test_idx_balanced))
    
    return balanced_splits

# Tutto il resto

In [74]:
if USE_MANUAL_5050_TEST:
    print("\n>>> USO TEST SET MANUALE 50/50")
    cv_splits = create_manual_5050_splits(X, y, n_splits=5, random_state=42)
else:
    print("\n>>> USO STRATIFIED K-FOLD + POST-HOC BALANCING")
    
    # Step 1: Crea split standard
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_splits_raw = list(skf.split(X, y))
    
    # Step 2: Bilancia i test set
    print("\n" + "="*60)
    print("BILANCIAMENTO POST-HOC DEI TEST SET")
    print("="*60 + "\n")
    cv_splits = balance_test_sets_post_hoc(cv_splits_raw, y)
    print(" Split bilanciati creati con successo\n")


# ========================================================================
# TRAINING E VALUTAZIONE
# ========================================================================

acc_scores, bal_scores, f1_scores, auc_scores = [], [], [], []  
results_rows = []

for fold, (train_idx, test_idx) in enumerate(cv_splits, start=1):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_test  = y.iloc[test_idx]

    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # ====================================================
    # VERIFICA BILANCIAMENTO TEST SET
    # ====================================================
    
    n_neg_test = (y_test == 0).sum()
    n_pos_test = (y_test == 1).sum()
    
    if not USE_MANUAL_5050_TEST:
        # Verifica che il post-hoc balancing abbia funzionato
        assert n_neg_test == n_pos_test, f" Test set NON bilanciato! (0={n_neg_test}, 1={n_pos_test})"
        print(f" Test set perfettamente bilanciato: {n_neg_test} vs {n_pos_test}")

    # ====================================================
    # SCALE_POS_WEIGHT
    # ====================================================

    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

    # DISTRIBUZIONE CLASSI
    print(f"\n BILANCIAMENTO CLASSI")
    print(f"{'-'*60}")
    print(f"TRAINING SET:")
    print(f"  Classe 0 (negativi): {n_neg:3d} campioni ({n_neg/(n_neg+n_pos)*100:5.1f}%)")
    print(f"  Classe 1 (positivi): {n_pos:3d} campioni ({n_pos/(n_neg+n_pos)*100:5.1f}%)")
    print(f"  Totale:              {n_neg + n_pos:3d} campioni")
    
    ratio = max(n_neg, n_pos) / min(n_neg, n_pos)
    print(f"\n  Ratio sbilanciamento: {ratio:.2f}x")
    print(f"  scale_pos_weight:     {scale_pos_weight:.4f}")
    
    # Interpretazione del peso
    if scale_pos_weight > 1.0:
        print(f"  → Classe 1 (pos) pesata {scale_pos_weight:.2f}x rispetto a classe 0")
    elif scale_pos_weight < 1.0:
        print(f"  → Classe 0 (neg) pesata {1/scale_pos_weight:.2f}x rispetto a classe 1")
    else:
        print(f"  → Classi già bilanciate")
    
    # Warning per pesi estremi
    if scale_pos_weight > 5.0:
        print(f" WARNING: peso molto alto - possibile overfitting su classe 1")
    elif scale_pos_weight < 0.2:
        print(f" WARNING: peso molto basso - classe 1 potrebbe essere ignorata")
    
    # Distribuzione test set
    print(f"\nTEST SET:")
    print(f"  Classe 0 (negativi): {n_neg_test:3d} campioni ({n_neg_test/len(y_test)*100:5.1f}%)")
    print(f"  Classe 1 (positivi): {n_pos_test:3d} campioni ({n_pos_test/len(y_test)*100:5.1f}%)")
    print(f"  Totale:              {len(y_test):3d} campioni")
    print(f"{'-'*60}")

    # ====================================================
    # MODELLO
    # ====================================================

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        n_estimators=100,    
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        random_state=42,
        n_jobs=1
    )

    # Verifica parametri PRIMA del fit
    print(f"\n VERIFICA PARAMETRI:")
    print(f"scale_pos_weight configurato: {model.get_params()['scale_pos_weight']:.4f}")

    model.fit(X_train, y_train)

    # Verifica parametri DOPO il fit
    print(f"scale_pos_weight effettivo:   {model.get_xgb_params()['scale_pos_weight']:.4f}")

    # ====================================================
    # DEBUG: PERFORMANCE SU TRAINING SET
    # ====================================================

    y_train_pred = model.predict(X_train)
    cm_train = confusion_matrix(y_train, y_train_pred)
    tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
    
    recall_0_train = tn_tr / (tn_tr + fp_tr) if (tn_tr + fp_tr) > 0 else 0
    recall_1_train = tp_tr / (fn_tr + tp_tr) if (fn_tr + tp_tr) > 0 else 0
    
    print(f"\n PERFORMANCE SU TRAINING SET (sanity check):")
    print(f"  Classe 0: {tn_tr}/{tn_tr+fp_tr} corretti ({recall_0_train*100:5.1f}%)")
    print(f"  Classe 1: {tp_tr}/{fn_tr+tp_tr} corretti ({recall_1_train*100:5.1f}%)")
    
    diff_train = abs(recall_0_train - recall_1_train)
    if diff_train < 0.1:
        print(f" Bilanciamento efficace (diff={diff_train:.3f})")
    elif diff_train < 0.2:
        print(f"Lievemente sbilanciato (diff={diff_train:.3f})")
    else:
        print(f"Ancora molto sbilanciato (diff={diff_train:.3f})")

    # ====================================================
    # FEATURE IMPORTANCE
    # ====================================================

    importances = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    print(f"\n TOP 20 FEATURE IMPORTANCE:")
    display(fi_df.head(20))

   
    fi_df.to_csv(f"feature_importance_fold_{fold}.csv", index=False)

    # ====================================================
    # PREDICTIONS SU TEST SET
    # ====================================================

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    # ====================================================
    # DEBUG: DISTRIBUZIONE PROBABILITÀ
    # ====================================================

    print(f"\n DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):")
    print(f"  Min:    {y_prob.min():.3f}")
    print(f"  25%:    {np.percentile(y_prob, 25):.3f}")
    print(f"  Median: {np.median(y_prob):.3f}")
    print(f"  75%:    {np.percentile(y_prob, 75):.3f}")
    print(f"  Max:    {y_prob.max():.3f}")
    
    # Predizioni incerte (vicine a 0.5)
    uncertain = np.sum((y_prob > 0.4) & (y_prob < 0.6))
    print(f"\n  Predizioni incerte (0.4-0.6): {uncertain}/{len(y_prob)} ({uncertain/len(y_prob)*100:.1f}%)")
    
    # Distribuzione per classe vera
    if len(y_test) > 0:
        prob_class_0 = y_prob[y_test == 0]
        prob_class_1 = y_prob[y_test == 1]
        
        if len(prob_class_0) > 0:
            print(f"\n  Classe 0 (vera negativa):")
            print(f"    Probabilità media: {prob_class_0.mean():.3f}")
            print(f"    Predette come 0:   {np.sum(prob_class_0 < 0.5)}/{len(prob_class_0)}")
        
        if len(prob_class_1) > 0:
            print(f"\n  Classe 1 (vera positiva):")
            print(f"    Probabilità media: {prob_class_1.mean():.3f}")
            print(f"    Predette come 1:   {np.sum(prob_class_1 >= 0.5)}/{len(prob_class_1)}")

    # ====================================================
    # METRICHE FINALI TEST SET
    # ====================================================

    acc = accuracy_score(y_test, y_pred)
    bal = balanced_accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    acc_scores.append(acc)
    bal_scores.append(bal)
    f1_scores.append(f1)
    auc_scores.append(auc)

    print(f"\n{'='*60}")
    print(f"METRICHE TEST SET - FOLD {fold}")
    print(f"{'='*60}")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {bal:.4f}")
    print(f"Differenza:        {abs(acc - bal):.3f}")
    
    # Verifica ACC = BAL_ACC
    if abs(acc - bal) < 1e-10:
        print(" ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)")
    elif abs(acc - bal) < 0.001:
        print(" ACC ≈ BAL_ACC (diff < 0.1%)")
    
    print(f"F1-score:          {f1:.4f}")
    print(f"ROC-AUC:           {auc:.4f}")

    print("Classification Report:")

    print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"\nTN={tn} | FP={fp} | FN={fn} | TP={tp}")
    
    # Breakdown per classe
    if (tn + fp) > 0:
        print(f"Classe 0: {tn}/{tn+fp} corretti ({tn/(tn+fp)*100:.1f}%)")
    if (fn + tp) > 0:
        print(f"Classe 1: {tp}/{fn+tp} corretti ({tp/(fn+tp)*100:.1f}%)")

    print(f"{'='*60}\n")

# ====================================================
# CALCOLO METRICHE FINALI
# ====================================================

mean_acc = np.mean(acc_scores)
mean_bal = np.mean(bal_scores)
mean_f1 = np.mean(f1_scores)
mean_auc = np.mean(auc_scores)

std_acc = np.std(acc_scores)
std_bal = np.std(bal_scores)
std_f1 = np.std(f1_scores)
std_auc = np.std(auc_scores)

diff_final = abs(mean_acc - mean_bal)

print("\n" + "="*60)
if USE_MANUAL_5050_TEST:
    print("RISULTATI FINALI - TEST SET MANUALE 50/50")
else:
    print("RISULTATI FINALI - STRATIFIED KFOLD + POST-HOC BALANCING")
print("="*60)
print(f"Accuracy:          {mean_acc:.3f} ± {std_acc:.3f}")
print(f"Balanced Accuracy: {mean_bal:.3f} ± {std_bal:.3f}")
print(f"Differenza:        {diff_final:.3f}") 

if diff_final < 1e-10:
    print(" ACC = BAL_ACC PERFETTAMENTE (differenza < 1e-10)")
elif diff_final < 0.001:
    print(" ACC ≈ BAL_ACC (differenza < 0.1%)")
elif diff_final < 0.01:
    print("Piccola differenza (< 1%)")
else:
    print(f"Differenza: {diff_final:.4f}")

print(f"F1-score:          {mean_f1:.3f} ± {std_f1:.3f}")
print(f"ROC-AUC:           {mean_auc:.3f} ± {std_auc:.3f}")
print("="*60)

print("\n")


>>> USO STRATIFIED K-FOLD + POST-HOC BALANCING

BILANCIAMENTO POST-HOC DEI TEST SET

Fold 1:
  PRIMA  - Test: Classe 0=10, Classe 1=7
  DOPO   - Test: Classe 0=7, Classe 1=7 (totale=14)
         - Train: 68 campioni

Fold 2:
  PRIMA  - Test: Classe 0=9, Classe 1=8
  DOPO   - Test: Classe 0=8, Classe 1=8 (totale=16)
         - Train: 66 campioni

Fold 3:
  PRIMA  - Test: Classe 0=9, Classe 1=7
  DOPO   - Test: Classe 0=7, Classe 1=7 (totale=14)
         - Train: 68 campioni

Fold 4:
  PRIMA  - Test: Classe 0=9, Classe 1=7
  DOPO   - Test: Classe 0=7, Classe 1=7 (totale=14)
         - Train: 68 campioni

Fold 5:
  PRIMA  - Test: Classe 0=9, Classe 1=7
  DOPO   - Test: Classe 0=7, Classe 1=7 (totale=14)
         - Train: 68 campioni

 Split bilanciati creati con successo


FOLD 1
 Test set perfettamente bilanciato: 7 vs 7

 BILANCIAMENTO CLASSI
------------------------------------------------------------
TRAINING SET:
  Classe 0 (negativi):  39 campioni ( 57.4%)
  Classe 1 (positivi):  2

,Feature,Importance
84,original_glszm_GrayLevelNonUniformity,0.057630
14,original_firstorder_10Percentile,0.033524
85,original_glszm_GrayLevelNonUniformityNormalized,0.032715
9,original_shape_MinorAxisLength,0.032490
10,original_shape_Sphericity,0.032129
91,original_glszm_LowGrayLevelZoneEmphasis,0.029684
82,original_glrlm_ShortRunHighGrayLevelEmphasis,0.029356
70,original_glrlm_GrayLevelVariance,0.025109
86,original_glszm_GrayLevelVariance,0.022063
4,original_shape_Maximum2DDiameterColumn,0.021651



 DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):
  Min:    0.053
  25%:    0.188
  Median: 0.351
  75%:    0.673
  Max:    0.849

  Predizioni incerte (0.4-0.6): 2/14 (14.3%)

  Classe 0 (vera negativa):
    Probabilità media: 0.370
    Predette come 0:   5/7

  Classe 1 (vera positiva):
    Probabilità media: 0.468
    Predette come 1:   4/7

METRICHE TEST SET - FOLD 1
Accuracy:          0.6429
Balanced Accuracy: 0.6429
Differenza:        0.000
 ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)
F1-score:          0.6154
ROC-AUC:           0.5714
Classification Report:

              precision    recall  f1-score   support

           0       0.62      0.71      0.67         7
           1       0.67      0.57      0.62         7

    accuracy                           0.64        14
   macro avg       0.65      0.64      0.64        14
weighted avg       0.65      0.64      0.64        14

Confusion Matrix:
[[5 2]
 [3 4]]

TN=5 | FP=2 | FN=3 | TP=4
Classe 0: 5/7 corretti (71.4%)
Classe 1: 4/

,Feature,Importance
96,original_glszm_SmallAreaLowGrayLevelEmphasis,0.052421
44,original_glcm_Imc1,0.040069
50,original_glcm_MCC,0.038425
26,original_firstorder_RootMeanSquared,0.037568
89,original_glszm_LargeAreaHighGrayLevelEmphasis,0.035783
95,original_glszm_SmallAreaHighGrayLevelEmphasis,0.032024
104,original_ngtdm_Strength,0.030637
53,original_glcm_SumEntropy,0.028865
57,original_gldm_DependenceNonUniformityNormalized,0.028485
2,original_shape_LeastAxisLength,0.024977



 DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):
  Min:    0.063
  25%:    0.173
  Median: 0.291
  75%:    0.520
  Max:    0.845

  Predizioni incerte (0.4-0.6): 4/16 (25.0%)

  Classe 0 (vera negativa):
    Probabilità media: 0.293
    Predette come 0:   5/8

  Classe 1 (vera positiva):
    Probabilità media: 0.407
    Predette come 1:   3/8

METRICHE TEST SET - FOLD 2
Accuracy:          0.5000
Balanced Accuracy: 0.5000
Differenza:        0.000
 ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)
F1-score:          0.4286
ROC-AUC:           0.6250
Classification Report:

              precision    recall  f1-score   support

           0       0.50      0.62      0.56         8
           1       0.50      0.38      0.43         8

    accuracy                           0.50        16
   macro avg       0.50      0.50      0.49        16
weighted avg       0.50      0.50      0.49        16

Confusion Matrix:
[[5 3]
 [5 3]]

TN=5 | FP=3 | FN=5 | TP=3
Classe 0: 5/8 corretti (62.5%)
Classe 1: 3/

...


 DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):
  Min:    0.027
  25%:    0.200
  Median: 0.419
  75%:    0.490
  Max:    0.933

  Predizioni incerte (0.4-0.6): 7/14 (50.0%)

  Classe 0 (vera negativa):
    Probabilità media: 0.335
    Predette come 0:   6/7

  Classe 1 (vera positiva):
    Probabilità media: 0.429
    Predette come 1:   3/7

METRICHE TEST SET - FOLD 3
Accuracy:          0.6429
Balanced Accuracy: 0.6429
Differenza:        0.000
 ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)
F1-score:          0.5455
ROC-AUC:           0.7347
Classification Report:

              precision    recall  f1-score   support

           0       0.60      0.86      0.71         7
           1       0.75      0.43      0.55         7

    accuracy                           0.64        14
   macro avg       0.68      0.64      0.63        14
weighted avg       0.67      0.64      0.63        14

Confusion Matrix:
[[6 1]
 [4 3]]

TN=6 | FP=1 | FN=4 | TP=3
Classe 0: 6/7 corretti (85.7%)
Classe 1: 3/

,Feature,Importance
7,original_shape_Maximum3DDiameter,0.064789
100,original_ngtdm_Busyness,0.055473
76,original_glrlm_RunEntropy,0.046570
61,original_gldm_LargeDependenceEmphasis,0.037273
10,original_shape_Sphericity,0.031825
86,original_glszm_GrayLevelVariance,0.028423
50,original_glcm_MCC,0.027462
91,original_glszm_LowGrayLevelZoneEmphasis,0.027254
96,original_glszm_SmallAreaLowGrayLevelEmphasis,0.026384
1,original_shape_Flatness,0.026116



 DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):
  Min:    0.120
  25%:    0.400
  Median: 0.635
  75%:    0.695
  Max:    0.831

  Predizioni incerte (0.4-0.6): 2/14 (14.3%)

  Classe 0 (vera negativa):
    Probabilità media: 0.519
    Predette come 0:   3/7

  Classe 1 (vera positiva):
    Probabilità media: 0.586
    Predette come 1:   5/7

METRICHE TEST SET - FOLD 4
Accuracy:          0.5714
Balanced Accuracy: 0.5714
Differenza:        0.000
 ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)
F1-score:          0.6250
ROC-AUC:           0.5306
Classification Report:

              precision    recall  f1-score   support

           0       0.60      0.43      0.50         7
           1       0.56      0.71      0.62         7

    accuracy                           0.57        14
   macro avg       0.58      0.57      0.56        14
weighted avg       0.58      0.57      0.56        14

Confusion Matrix:
[[3 4]
 [2 5]]

TN=3 | FP=4 | FN=2 | TP=5
Classe 0: 3/7 corretti (42.9%)
Classe 1: 5/

,Feature,Importance
48,original_glcm_JointEnergy,0.041179
2,original_shape_LeastAxisLength,0.033760
100,original_ngtdm_Busyness,0.033322
95,original_glszm_SmallAreaHighGrayLevelEmphasis,0.029900
91,original_glszm_LowGrayLevelZoneEmphasis,0.029539
104,original_ngtdm_Strength,0.027953
65,original_gldm_SmallDependenceEmphasis,0.027921
86,original_glszm_GrayLevelVariance,0.025256
103,original_ngtdm_Contrast,0.024836
10,original_shape_Sphericity,0.024369



 DISTRIBUZIONE PROBABILITÀ PREDETTE (test set):
  Min:    0.104
  25%:    0.293
  Median: 0.555
  75%:    0.701
  Max:    0.848

  Predizioni incerte (0.4-0.6): 1/14 (7.1%)

  Classe 0 (vera negativa):
    Probabilità media: 0.391
    Predette come 0:   5/7

  Classe 1 (vera positiva):
    Probabilità media: 0.600
    Predette come 1:   5/7

METRICHE TEST SET - FOLD 5
Accuracy:          0.7143
Balanced Accuracy: 0.7143
Differenza:        0.000
 ACC = BAL_ACC PERFETTAMENTE (diff < 1e-10)
F1-score:          0.7143
ROC-AUC:           0.7143
Classification Report:

              precision    recall  f1-score   support

           0       0.71      0.71      0.71         7
           1       0.71      0.71      0.71         7

    accuracy                           0.71        14
   macro avg       0.71      0.71      0.71        14
weighted avg       0.71      0.71      0.71        14

Confusion Matrix:
[[5 2]
 [2 5]]

TN=5 | FP=2 | FN=2 | TP=5
Classe 0: 5/7 corretti (71.4%)
Classe 1: 5/7

# CSV


In [75]:
row = {
    "Dataset": "AMBL",
    "Target": "PR",
    "Accuracy": f"{mean_acc:.3f} ± {std_acc:.3f}",
    "Balanced Accuracy": f"{mean_bal:.3f} ± {std_bal:.3f}",
    "Differenza": f"{diff_final:.3f}",
    "F1-score": f"{mean_f1:.3f} ± {std_f1:.3f}",
    "ROC-AUC": f"{mean_auc:.3f} ± {std_auc:.3f}"
}

results_df = pd.DataFrame([row])
display(results_df)

,Dataset,Target,Accuracy,Balanced Accuracy,Differenza,F1-score,ROC-AUC
0,AMBL,PR,0.614 ± 0.073,0.614 ± 0.073,0.000,0.586 ± 0.095,0.635 ± 0.079
